# Module 04 AI Workbook — solution and explanation

> Try the exercise yourself before reading this.

## The adversarial bug

[../adversarial_histogram_buggy.py](./adversarial_histogram_buggy.py) fills the
histogram with:

```python
hist[idx] += 1        # idx has many repeated values
```

NumPy evaluates this as `hist[idx] = hist[idx] + 1`. The right-hand side is
computed against the *original* `hist`, and the assignment writes each distinct
index once. So if bin 25 appears 4000 times in `idx`, it still ends up
incremented by **1**, not 4000. The histogram undercounts massively.

This is the exact CPU mirror of a **GPU histogram without `atomicAdd`**: many
threads doing `hist[b] += 1` into the same bin read-modify-write concurrently and
clobber each other, losing counts.

## The fix

Use an accumulating scatter-add:

```python
np.add.at(hist, idx, 1)              # CPU: unbuffered, accumulates duplicates
# or simply
hist = np.bincount(idx, minlength=nbins)
```

On the GPU the equivalent is an atomic:

```python
cuda.atomic.add(hist, b, 1)          # Numba CUDA
```

Both are in [histogram_solution.py](solutions/histogram_solution.py). The GPU path runs
only if `numba` and a CUDA device are present; otherwise it prints a skip so the
file still runs on a CPU-only machine.

## Why the verification matters

Checking the histogram's *shape* or that it "looks bell-shaped" would not catch
this — the shape is fine, only the magnitudes are wrong. The reliable test is
`hist.sum() == n_events` **and** a bin-by-bin comparison against `np.bincount`,
which is what [../verify_histogram.py](./verify_histogram.py) does.

## Mapping to the 5-phase loop

- **PREDICT:** flag `hist[idx] += 1` with duplicate indices as a scatter race.
- **VERIFY:** total-count and bin-by-bin comparison against the reference.
- **DIAGNOSE:** the fix is O(n) either way; on the GPU, atomics into few bins can
  serialize — the next optimization is per-block privatized histograms, mirroring
  shared-memory histogramming in CUDA.


## Run the worked solution

The cells below build and run the solution. Each should finish with `PASS`.

In [ ]:
!python solutions/histogram_solution.py